# 🏥 HealthBridge Medical Center
## Medication Feature Engineering — Condensing 23 Columns Into Usable Model Features

---

### The Problem
The clean file has **23 medication columns**, but many are nearly useless:
- `examide`, `citoglipton`, `glimepiride-pioglitazone` → **100% 'No'** — zero variance, zero information
- `tolbutamide`, `troglitazone`, `tolazamide` → **<0.1% active** — statistically meaningless
- Most combo medications → **<1% active**

Feeding 23 raw columns into a model adds noise and bloat. Instead we engineer **4 focused, meaningful features** that capture the clinical signal without the clutter.

### What This Notebook Creates

| New Feature | What It Measures | Why It Matters |
|---|---|---|
| `med_count` | How many medications are active | Polypharmacy → higher readmission risk |
| `med_changed_count` | How many meds had dose Up or Down | Dosage instability at discharge |
| `insulin_active` | Is the patient on insulin (0/1) | Strongest single medication signal (51% active) |
| `key_med_active` | Is any high-signal med active (0/1) | Covers the 5 most clinically significant drugs |

The 23 raw columns are then replaced by these 4 engineered features + the existing `change` and `diabetesMed` columns.

## CELL 1 — Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# ── Load clean file ───────────────────────────────────────────────────────────
# Update path if using Google Drive:
# df = pd.read_excel('/content/drive/MyDrive/HealthBridge_Project/diabetic_clean.xlsx')
df = pd.read_excel('diabetic_clean.xlsx', engine='openpyxl')

# Create binary target
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Positive class: {df["readmitted_30"].mean()*100:.1f}%')

## CELL 2 — Audit All 23 Medication Columns

**What:** Print a usage summary for every medication column.

**Why:** Before condensing, you need to see exactly which medications have real signal and which are essentially empty. This audit drives every decision in the next steps.

In [ ]:
# All 23 medication columns in the dataset
all_med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]
med_cols = [c for c in all_med_cols if c in df.columns]

print(f'Medication columns found: {len(med_cols)}')
print()

# Build summary table
rows = []
for col in med_cols:
    vc = df[col].value_counts()
    total_active  = (df[col] != 'No').sum()
    pct_active    = total_active / len(df) * 100
    n_steady      = vc.get('Steady', 0)
    n_up          = vc.get('Up', 0)
    n_down        = vc.get('Down', 0)
    n_unique_vals = df[col].nunique()

    if pct_active == 0:
        action = 'DROP — zero variance'
    elif pct_active < 0.5:
        action = 'DROP — <0.5% active'
    elif pct_active < 5:
        action = 'INCLUDE in med_count'
    else:
        action = 'KEY SIGNAL — keep individually'

    rows.append({
        'Medication': col,
        'Total Active': total_active,
        '% Active': round(pct_active, 1),
        'Steady': n_steady,
        'Up': n_up,
        'Down': n_down,
        'Unique Values': n_unique_vals,
        'Action': action
    })

audit_df = pd.DataFrame(rows).sort_values('% Active', ascending=False)
print(audit_df.to_string(index=False))

## CELL 3 — Visualize Medication Usage

**What:** Bar chart showing the % of patients on each medication.

**Why:** Makes the skewness obvious — insulin dominates, most others are near-zero. This visualization is useful for your presentation to justify why you condensed the columns.

In [ ]:
# Calculate % active for each medication
med_usage = {}
for col in med_cols:
    pct = (df[col] != 'No').mean() * 100
    med_usage[col] = pct

usage_series = pd.Series(med_usage).sort_values(ascending=True)

# Color code: green = keep individually, orange = include in count, red = drop
bar_colors = []
for pct in usage_series.values:
    if pct == 0:   bar_colors.append('#C00000')   # red  — drop
    elif pct < 0.5: bar_colors.append('#E87722')  # orange — drop
    elif pct < 5:  bar_colors.append('#D4860A')   # amber — include in count
    else:          bar_colors.append('#1D7A4F')   # green — key signal

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(usage_series.index, usage_series.values,
               color=bar_colors, edgecolor='black', linewidth=0.5)
ax.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='0.5% threshold')
ax.axvline(5.0, color='green',  linestyle='--', linewidth=1.5, label='5% threshold (key signal)')
ax.set_xlabel('% of Patients with Active/Changed Prescription')
ax.set_title('Medication Usage Across 69,990 Patients\n(Green = key signal | Amber = include in count | Red = drop)',
             fontweight='bold')
ax.legend()

for bar, pct in zip(bars, usage_series.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('medication_usage_audit.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMedications to DROP (0% active):         {sum(1 for p in usage_series.values if p == 0)}')
print(f'Medications to DROP (<0.5% active):      {sum(1 for p in usage_series.values if 0 < p < 0.5)}')
print(f'Medications to count (0.5%–5% active):   {sum(1 for p in usage_series.values if 0.5 <= p < 5)}')
print(f'Key signal medications (5%+ active):     {sum(1 for p in usage_series.values if p >= 5)}')

## CELL 4 — Engineer Feature 1: `med_count`

**What:** Count how many medications each patient has active (Steady, Up, or Down — anything except 'No').

**Why:** This captures **polypharmacy** — the clinical phenomenon where patients on many medications simultaneously are more complex to manage and harder to keep stable after discharge. A single number is far more useful to a model than 23 separate binary flags.

**Only counts medications with any real usage** — zero-variance columns are excluded.

In [ ]:
# Medications with at least some real usage (>0% active)
# Exclude: examide, citoglipton, glimepiride-pioglitazone (100% 'No')
countable_meds = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol',
    'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]
countable_meds = [c for c in countable_meds if c in df.columns]

# Count how many are NOT 'No' for each patient
df['med_count'] = df[countable_meds].apply(
    lambda row: (row != 'No').sum(), axis=1
)

print('med_count distribution:')
print(df['med_count'].value_counts().sort_index().to_string())
print(f'\nMean active medications per patient: {df["med_count"].mean():.2f}')
print(f'Max active medications:              {df["med_count"].max()}')

# Quick validation — check against the readmission rate
df['med_count_capped'] = df['med_count'].clip(upper=5)
readmit_by_med = df.groupby('med_count_capped')['readmitted_30'].mean().mul(100).round(1)
print('\n30-day readmission rate by med_count:')
for cnt, rate in readmit_by_med.items():
    label = str(cnt) if cnt < 5 else '5+'
    print(f'  {label} active meds → {rate:.1f}% readmission rate')

## CELL 5 — Engineer Feature 2: `med_changed_count`

**What:** Count how many medications had their dosage changed (Up or Down) — not just active but actually adjusted.

**Why:** A dosage change at or before discharge signals **medication instability**. A patient whose insulin was increased ('Up') or whose metformin was reduced ('Down') is harder to stabilize at home than one on a steady regimen. This is a clinically meaningful signal that gets lost when you just encode Steady/Up/Down as a single 0/1 flag.

In [ ]:
# Count meds where dosage was CHANGED (Up or Down)
df['med_changed_count'] = df[countable_meds].apply(
    lambda row: ((row == 'Up') | (row == 'Down')).sum(), axis=1
)

print('med_changed_count distribution:')
print(df['med_changed_count'].value_counts().sort_index().to_string())
print(f'\nPatients with at least one dosage change: {(df["med_changed_count"] > 0).sum():,} '
      f'({(df["med_changed_count"] > 0).mean()*100:.1f}%)')

# Readmission rate by dosage changes
df['changed_capped'] = df['med_changed_count'].clip(upper=3)
readmit_by_change = df.groupby('changed_capped')['readmitted_30'].mean().mul(100).round(1)
print('\n30-day readmission rate by med_changed_count:')
for cnt, rate in readmit_by_change.items():
    label = str(cnt) if cnt < 3 else '3+'
    print(f'  {label} meds changed → {rate:.1f}% readmission rate')

## CELL 6 — Engineer Feature 3: `insulin_active`

**What:** Binary flag — is the patient on insulin at all (1) or not (0)?

**Why:** Insulin is by far the most commonly prescribed medication in this dataset (51% of patients). It also has four possible values (No / Steady / Up / Down), making it the one medication complex enough to warrant its own feature. Patients on insulin are managing a more severe form of diabetes that requires tighter monitoring post-discharge.

In [ ]:
# Binary: 1 = any insulin (Steady, Up, or Down), 0 = No insulin
df['insulin_active'] = (df['insulin'] != 'No').astype(int)

print('insulin_active distribution:')
print(df['insulin_active'].value_counts())
print(f'\nPatients on insulin: {df["insulin_active"].sum():,} ({df["insulin_active"].mean()*100:.1f}%)')

# Readmission rate breakdown by insulin status
insulin_readmit = df.groupby('insulin')['readmitted_30'].agg(['mean','count'])
insulin_readmit.columns = ['Readmit Rate', 'Count']
insulin_readmit['Readmit Rate'] = insulin_readmit['Readmit Rate'].mul(100).round(1)
print('\nReadmission rate by insulin status:')
print(insulin_readmit.to_string())

overall_rate = df['readmitted_30'].mean() * 100
on_insulin_rate = df[df['insulin_active']==1]['readmitted_30'].mean() * 100
off_insulin_rate = df[df['insulin_active']==0]['readmitted_30'].mean() * 100
print(f'\nOn insulin:  {on_insulin_rate:.1f}%  vs  Overall: {overall_rate:.1f}%')
print(f'Off insulin: {off_insulin_rate:.1f}%')

## CELL 7 — Engineer Feature 4: `key_med_active`

**What:** Binary flag — is the patient on ANY of the five highest-usage, most clinically significant diabetes medications?

**Why:** The five key medications (insulin, metformin, glipizide, glyburide, pioglitazone) together cover the vast majority of active prescriptions in this dataset. A single flag for 'on at least one key medication' is more informative than individual binary columns for each, especially since most patients on any one of these are likely on others too.

In [ ]:
# The 5 medications with 5%+ active patients — the key signal group
key_meds = ['insulin', 'metformin', 'glipizide', 'glyburide', 'pioglitazone']
key_meds = [c for c in key_meds if c in df.columns]

df['key_med_active'] = df[key_meds].apply(
    lambda row: int((row != 'No').any()), axis=1
)

print('key_med_active distribution:')
print(df['key_med_active'].value_counts())

key_readmit_rate  = df[df['key_med_active']==1]['readmitted_30'].mean() * 100
no_key_readmit_rate = df[df['key_med_active']==0]['readmitted_30'].mean() * 100
print(f'\nOn key medication:     {key_readmit_rate:.1f}% readmission rate')
print(f'Not on key medication: {no_key_readmit_rate:.1f}% readmission rate')
print(f'Overall average:       {overall_rate:.1f}%')

# Also add a count of key meds specifically
df['key_med_count'] = df[key_meds].apply(
    lambda row: (row != 'No').sum(), axis=1
)
print('\nDistribution of how many key meds are active per patient:')
print(df['key_med_count'].value_counts().sort_index())

## CELL 8 — Visualize All 4 Engineered Features vs Readmission

**What:** Side-by-side charts showing how each new feature relates to 30-day readmission rate.

**Why:** This is your validation step — if the engineered features show a meaningful relationship with the target variable, they are earning their place in the model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
C_BLUE = '#2E75B6'
C_RED  = '#C00000'

overall_rate = df['readmitted_30'].mean() * 100

# ── Chart 1: Readmission rate by med_count ────────────────────────────────────
mc_capped = df['med_count'].clip(upper=6)
mc_rate = df.assign(mc_cap=mc_capped).groupby('mc_cap')['readmitted_30'].agg(['mean','count'])
mc_rate['rate_pct'] = mc_rate['mean'] * 100
tick_labels = {i: str(i) if i < 6 else '6+' for i in mc_rate.index}
bar_cols = [C_RED if r > overall_rate else C_BLUE for r in mc_rate['rate_pct']]
bars1 = axes[0,0].bar([tick_labels[i] for i in mc_rate.index], mc_rate['rate_pct'],
                       color=bar_cols, edgecolor='black', linewidth=0.5)
axes[0,0].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                  label=f'Overall avg: {overall_rate:.1f}%')
axes[0,0].set_title('Readmission Rate by med_count\n(# active medications)')
axes[0,0].set_xlabel('Number of Active Medications')
axes[0,0].set_ylabel('30-Day Readmission Rate (%)')
axes[0,0].legend(fontsize=9)
for bar, (_, row) in zip(bars1, mc_rate.iterrows()):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                   f'{row["rate_pct"]:.1f}%', ha='center', fontsize=8)

# ── Chart 2: Readmission rate by med_changed_count ────────────────────────────
mcc_capped = df['med_changed_count'].clip(upper=3)
mcc_rate = df.assign(mcc_cap=mcc_capped).groupby('mcc_cap')['readmitted_30'].agg(['mean','count'])
mcc_rate['rate_pct'] = mcc_rate['mean'] * 100
tick_labels2 = {i: str(i) if i < 3 else '3+' for i in mcc_rate.index}
bar_cols2 = [C_RED if r > overall_rate else C_BLUE for r in mcc_rate['rate_pct']]
bars2 = axes[0,1].bar([tick_labels2[i] for i in mcc_rate.index], mcc_rate['rate_pct'],
                       color=bar_cols2, edgecolor='black', linewidth=0.5)
axes[0,1].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                  label=f'Overall avg: {overall_rate:.1f}%')
axes[0,1].set_title('Readmission Rate by med_changed_count\n(# dosages changed Up or Down)')
axes[0,1].set_xlabel('Number of Medication Dosage Changes')
axes[0,1].set_ylabel('30-Day Readmission Rate (%)')
axes[0,1].legend(fontsize=9)
for bar, (_, row) in zip(bars2, mcc_rate.iterrows()):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                   f'{row["rate_pct"]:.1f}%', ha='center', fontsize=8)

# ── Chart 3: Readmission rate by insulin_active ───────────────────────────────
ins_rate = df.groupby('insulin_active')['readmitted_30'].agg(['mean','count'])
ins_rate['rate_pct'] = ins_rate['mean'] * 100
bars3 = axes[1,0].bar(['No Insulin (0)', 'On Insulin (1)'], ins_rate['rate_pct'],
                       color=[C_BLUE, C_RED], edgecolor='black', linewidth=0.5, width=0.4)
axes[1,0].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                  label=f'Overall avg: {overall_rate:.1f}%')
axes[1,0].set_title('Readmission Rate by insulin_active')
axes[1,0].set_ylabel('30-Day Readmission Rate (%)')
axes[1,0].legend(fontsize=9)
for bar, (_, row) in zip(bars3, ins_rate.iterrows()):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                   f'{row["rate_pct"]:.1f}%\n(n={int(row["count"]):,})',
                   ha='center', fontsize=9)

# ── Chart 4: Readmission rate by key_med_active ───────────────────────────────
km_rate = df.groupby('key_med_active')['readmitted_30'].agg(['mean','count'])
km_rate['rate_pct'] = km_rate['mean'] * 100
bars4 = axes[1,1].bar(['No Key Med (0)', 'On Key Med (1)'], km_rate['rate_pct'],
                       color=[C_BLUE, C_RED], edgecolor='black', linewidth=0.5, width=0.4)
axes[1,1].axhline(overall_rate, color='navy', linestyle='--', linewidth=1.5,
                  label=f'Overall avg: {overall_rate:.1f}%')
axes[1,1].set_title('Readmission Rate by key_med_active\n(insulin / metformin / glipizide / glyburide / pioglitazone)')
axes[1,1].set_ylabel('30-Day Readmission Rate (%)')
axes[1,1].legend(fontsize=9)
for bar, (_, row) in zip(bars4, km_rate.iterrows()):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                   f'{row["rate_pct"]:.1f}%\n(n={int(row["count"]):,})',
                   ha='center', fontsize=9)

plt.suptitle('Engineered Medication Features vs 30-Day Readmission Rate',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('medication_features_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 9 — Build the Final Condensed Medication Feature Set

**What:** Drop the 23 raw medication columns and keep only the 6 engineered/retained features.

**Why:** Replacing 23 mostly-empty columns with 6 meaningful ones reduces dimensionality, removes noise from zero-variance columns, and gives the model cleaner, more interpretable signals to learn from.

In [ ]:
print('Columns BEFORE condensing medication features:')
print(f'  Total: {df.shape[1]} columns')
print(f'  Medication columns: {len([c for c in all_med_cols if c in df.columns])}')
print()

# ── Drop all 23 original medication columns ───────────────────────────────────
df_condensed = df.drop(columns=[c for c in all_med_cols if c in df.columns])

# ── The 6 medication features we keep ─────────────────────────────────────────
# Already created in Cells 4-7:
#   med_count           — total active medications per patient
#   med_changed_count   — total dosage changes (Up or Down)
#   insulin_active      — binary: on insulin (1) or not (0)
#   key_med_active      — binary: on any key medication (1) or not (0)
#   key_med_count       — count of key medications active
# Also keep from original:
#   change              — overall medication change flag (Ch / No)
#   diabetesMed         — any diabetes medication at all (Yes / No)

# Encode the two remaining categorical medication columns
df_condensed['change_enc']      = (df_condensed['change'] == 'Ch').astype(int)
df_condensed['diabetesMed_enc'] = (df_condensed['diabetesMed'] == 'Yes').astype(int)
df_condensed = df_condensed.drop(columns=['change', 'diabetesMed'])

# Drop the temp capped columns used for visualization
for col in ['med_count_capped', 'changed_capped', 'inpatient_capped', 'diag_1_cat']:
    if col in df_condensed.columns:
        df_condensed = df_condensed.drop(columns=[col])

print('Columns AFTER condensing medication features:')
print(f'  Total: {df_condensed.shape[1]} columns')
print(f'  Medication-related features: 7 (med_count, med_changed_count, insulin_active,')
print(f'                                  key_med_active, key_med_count, change_enc, diabetesMed_enc)')
print()
print('  Columns removed: 23  →  Columns added: 7  →  Net reduction: 16 columns')
print()
print('Full column list after condensing:')
for i, col in enumerate(df_condensed.columns, 1):
    flag = ' ← new med feature' if col in ['med_count','med_changed_count','insulin_active',
                                             'key_med_active','key_med_count',
                                             'change_enc','diabetesMed_enc'] else ''
    print(f'  {i:2}. {col}{flag}')

## CELL 10 — Correlation of New Features with Target

**What:** Check how strongly each new medication feature correlates with readmitted_30.

**Why:** Confirms the engineered features are actually informative before you pass them into the model. Any feature with near-zero correlation adds noise without adding signal.

In [ ]:
new_med_features = ['med_count', 'med_changed_count', 'insulin_active',
                    'key_med_active', 'key_med_count', 'change_enc', 'diabetesMed_enc']

# Add readmitted_30 if it was dropped
if 'readmitted_30' not in df_condensed.columns:
    df_condensed['readmitted_30'] = df['readmitted_30'].values

corr_vals = df_condensed[new_med_features + ['readmitted_30']].corr()['readmitted_30'].drop('readmitted_30')

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#C00000' if v > 0 else '#2E75B6' for v in corr_vals.values]
bars = ax.barh(corr_vals.index, corr_vals.values, color=bar_colors,
               edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation of Engineered Medication Features with readmitted_30',
             fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
for bar, val in zip(bars, corr_vals.values):
    ax.text(val + (0.001 if val >= 0 else -0.001), bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)
plt.tight_layout()
plt.savefig('medication_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlation with readmitted_30:')
for feat, val in corr_vals.abs().sort_values(ascending=False).items():
    direction = corr_vals[feat]
    print(f'  {feat:<25} {direction:+.4f}  {"↑ positive relationship" if direction > 0 else "↓ negative relationship"}')

## CELL 11 — Save the Condensed File

**What:** Save the condensed dataset with engineered medication features as a new Excel file.

**Why:** This is the file you will load into your preprocessing notebook as the starting point. It replaces `diabetic_clean.xlsx` with a leaner, better-engineered version ready for encoding and modeling.

In [ ]:
# Final shape check
print('Final condensed dataset:')
print(f'  Shape:          {df_condensed.shape[0]:,} rows × {df_condensed.shape[1]} columns')
print(f'  Missing values: {df_condensed.isnull().sum().sum()}')
print(f'  Positive class: {df_condensed["readmitted_30"].mean()*100:.1f}%')
print()

# Save
df_condensed.to_excel('diabetic_condensed.xlsx', index=False, engine='openpyxl')
df_condensed.to_csv('diabetic_condensed.csv', index=False)
print('Saved:')
print('  diabetic_condensed.xlsx  ← use this as input to your preprocessing notebook')
print('  diabetic_condensed.csv   ← lighter version for Python pipelines')

# Download
from google.colab import files
files.download('diabetic_condensed.xlsx')
files.download('diabetic_condensed.csv')
print('\nDownloads triggered — check your Downloads folder')

## CELL 12 — Summary: What Changed

**What:** Print a clean before/after comparison of the medication feature strategy.

In [ ]:
print('=' * 65)
print('  MEDICATION FEATURE CONDENSING — SUMMARY')
print('=' * 65)
print()
print('BEFORE — 23 raw medication columns:')
print('  Problem 1: examide, citoglipton, glimepiride-pioglitazone,')
print('             metformin-rosiglitazone, metformin-pioglitazone')
print('             → 100% "No" — zero variance, pure noise')
print('  Problem 2: tolbutamide, troglitazone, tolazamide, miglitol')
print('             → <0.1% active — statistically meaningless')
print('  Problem 3: 23 columns to encode individually → bloats feature space')
print()
print('AFTER — 7 engineered medication features:')
print()

features_summary = [
    ('med_count',         'Numeric (0–7+)', 'Total active medications per patient'),
    ('med_changed_count', 'Numeric (0–3+)', 'Medications with dosage changed (Up or Down)'),
    ('insulin_active',    'Binary (0/1)',   'Patient on insulin — strongest single med signal'),
    ('key_med_active',    'Binary (0/1)',   'On any of 5 key meds (insulin/metformin/etc.)'),
    ('key_med_count',     'Numeric (0–5)',  'Count of key medications active simultaneously'),
    ('change_enc',        'Binary (0/1)',   'Any medication change at discharge (Ch=1)'),
    ('diabetesMed_enc',   'Binary (0/1)',   'Any diabetes medication prescribed (Yes=1)'),
]

for feat, dtype, desc in features_summary:
    print(f'  {feat:<22} {dtype:<18} {desc}')

print()
print(f'Columns removed: 23  |  Columns added: 7  |  Net: -16 columns')
print(f'Dataset shape:   (69,990 × 48)  →  (69,990 × {df_condensed.shape[1]})')
print()
print('NEXT STEP: Load diabetic_condensed.xlsx into your')
print('  preprocessing notebook and run the encoding + SMOTE pipeline.')
print('=' * 65)

---
## What to Update in Your Preprocessing Notebook

When you load `diabetic_condensed.xlsx` into your preprocessing notebook, update the medication encoding step:

**Before (old approach — encoding 23 raw columns):**
```python
med_cols = ['metformin', 'repaglinide', ... 'insulin', ...]
for col in med_cols:
    df[col + '_enc'] = (df[col] != 'No').astype(int)
```

**After (new approach — medication features already engineered):**
```python
# med_count, med_changed_count, insulin_active,
# key_med_active, key_med_count, change_enc, diabetesMed_enc
# are already numeric — no further encoding needed
# Just include them directly in your feature set X
```

---
*HealthBridge Medical Center | Advanced Data Analytics Capstone | AY 2025–2026*